# BirdCLEF+ 2026 — Submission Notebook V4 (3-Model R2 ONLY TTA) (6-Model Ensemble + Slice Fix)
## CPU Only | Internet OFF
### Gerekli Input'lar:
1. `birdclef-2026` — yarışma verisi
2. `birdclef2026-v7-models` — training notebook output'u (Kaggle Dataset olarak yükle)

### Training Output'unda olması gereken dosyalar:
- `best_r2_fold1.pth`, `best_r2_fold2.pth`, `best_r2_fold3.pth`
- `species_list.json`

In [1]:
import os, warnings, json
import numpy as np
import pandas as pd
import librosa
import cv2
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import timm

# ============ CONFIG ============
SAMPLE_RATE = 32000
DURATION    = 5
N_MELS      = 128
N_FFT       = 1024
HOP_LENGTH  = 512
IMG_SIZE    = 224
NUM_FOLDS   = 3
MODEL_NAME  = 'tf_efficientnetv2_s'
N_WINDOWS   = 3  # Multi-window TTA

# CPU only!
DEVICE = torch.device('cpu')

# Paths
BASE_DIR   = Path('/kaggle/input/competitions/birdclef-2026')
TEST_DIR   = BASE_DIR / 'test_soundscapes'
SUB_CSV    = BASE_DIR / 'sample_submission.csv'

# Training notebook output'unu Kaggle Dataset olarak eklemen lazim
# Asagidaki path'i kendi dataset adina gore degistir!
MODELS_DIR = Path('/kaggle/input/datasets/loxxy46/birdclef2026-v7-models')

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

print(f'Device: {DEVICE}')
print(f'Models dir: {MODELS_DIR}')
print(f'Test dir: {TEST_DIR}')

Device: cpu
Models dir: /kaggle/input/datasets/loxxy46/birdclef2026-v7-models
Test dir: /kaggle/input/competitions/birdclef-2026/test_soundscapes


In [2]:
# species_list.json yoksa, train_audio'dan oluştur
import json
from pathlib import Path

BASE_DIR = Path('/kaggle/input/competitions/birdclef-2026')
TRAIN_AUDIO = BASE_DIR / 'train_audio'

all_species = sorted([sp.name for sp in TRAIN_AUDIO.iterdir() if sp.is_dir()])
print(f'Species: {len(all_species)}')
print(f'First 5: {all_species[:5]}')
NUM_CLASSES = len(all_species)
species2idx = {sp: i for i, sp in enumerate(all_species)}
print(f'NUM_CLASSES: {NUM_CLASSES}')

Species: 206
First 5: ['1161364', '116570', '1176823', '1595929', '209233']
NUM_CLASSES: 206


In [3]:
# ============ MODEL DEFINITION ============
# Ayni model sinifi — training notebook ile AYNI olmali!
class AttentionPool(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features // 4),
            nn.Tanh(),
            nn.Linear(in_features // 4, 1))
    def forward(self, x):
        att_w = torch.softmax(self.attention(x), dim=1)
        return (att_w * x).sum(dim=1)

class BirdSED(nn.Module):
    def __init__(self, num_classes, model_name=MODEL_NAME):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False,
            num_classes=0, global_pool='')
        in_f = self.backbone.num_features
        self.att_pool = AttentionPool(in_f)
        self.head = nn.Sequential(
            nn.LayerNorm(in_f),
            nn.Dropout(0.35),
            nn.Linear(in_f, 512),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes))
    def forward(self, x):
        feat = self.backbone(x)
        B, C, H, W = feat.shape
        feat = feat.permute(0, 2, 3, 1).reshape(B, H*W, C)
        pooled = self.att_pool(feat)
        return self.head(pooled)

print('Model class defined.')

Model class defined.


In [4]:
# ============ LOAD ENSEMBLE MODELS (SADECE R2 - HIZ GUNCELLEMESI) ============
ensemble = []
# Onceden 6 Model de yukluyordu (R1 ve R2). CPU limitine takildigi icin 
# Sadece Pseudo-Labeling ile Fine-Tune edilmis en guclu 3 modeli (R2'leri) kullaniyoruz!
for fold in range(1, NUM_FOLDS + 1):
    for prefix in ['r2']:
        model_path = MODELS_DIR / f'best_{prefix}_fold{fold}.pth'
        if model_path.exists():
            m = BirdSED(NUM_CLASSES)
            m.load_state_dict(torch.load(str(model_path), map_location='cpu'))
            m.eval()
            ensemble.append(m)
            print(f'Loaded: {model_path.name} ({(model_path.stat().st_size/1024/1024):.1f} MB)')
        else:
            print(f'NOT FOUND: {model_path.name}')

print(f'\nEnsemble size: {len(ensemble)}')
assert len(ensemble) > 0, 'No models found! Check MODELS_DIR path.'


Loaded: best_r2_fold1.pth (82.3 MB)
Loaded: best_r2_fold2.pth (82.3 MB)
Loaded: best_r2_fold3.pth (82.3 MB)

Ensemble size: 3


In [5]:
# ============ INFERENCE FUNCTIONS (OPTIMIZED) ============
current_audio_id = None
current_audio = None

def get_audio_slice(audio_id, filepath, end_sec):
    global current_audio_id, current_audio
    
    # Kaggle gizli test setinde ayni ses dosyasindan arka arkaya (0-5, 5-10, 10-15...) 
    # tahmin istendigi icin dosyayi sadece BIR KERE RAM'e yukleyip hafizada tutuyoruz (Caching).
    # Eski yontemde dosyayi yuzlerce kez bastan okudugu icin Timeout oldu! Kacis yok :)
    if audio_id != current_audio_id:
        try:
            current_audio, _ = librosa.load(str(filepath), sr=SAMPLE_RATE, mono=True)
            current_audio_id = audio_id
        except:
            current_audio = np.zeros(SAMPLE_RATE * DURATION)
            current_audio_id = audio_id
            
    start_sec = max(0, float(end_sec) - DURATION)
    start_idx = int(start_sec * SAMPLE_RATE)
    end_idx = int(end_sec * SAMPLE_RATE)
    
    slice_audio = current_audio[start_idx : end_idx]
    
    target_len = int(SAMPLE_RATE * DURATION)
    if len(slice_audio) < target_len:
        slice_audio = np.pad(slice_audio, (0, target_len - len(slice_audio)))
    else:
        slice_audio = slice_audio[:target_len]
        
    return slice_audio

def slice_to_mel_tensor(slice_audio):
    """Audio Slice -> Mel Spectrogram -> Normalized Tensor"""
    mel = librosa.feature.melspectrogram(
        y=slice_audio, sr=SAMPLE_RATE,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=20, fmax=16000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    mel_u8 = (mel_norm * 255).astype(np.uint8)
    mel_res = cv2.resize(mel_u8, (IMG_SIZE, IMG_SIZE))
    mel_3ch = np.stack([mel_res]*3, axis=-1)
    mel_t = torch.FloatTensor(mel_3ch).permute(2, 0, 1).unsqueeze(0) / 255.0
    mel_t = (mel_t - MEAN.unsqueeze(0)) / STD.unsqueeze(0)
    return mel_t

print('Optimized Inference functions ready!')


Optimized Inference functions ready!


In [6]:
# ============ GENERATE SUBMISSION ============
sub = pd.read_csv(SUB_CSV)
print(f'Submission rows: {len(sub)}')
print(f'Columns: {list(sub.columns[:5])}...')

species_cols = [sp for sp in all_species if sp in sub.columns]
AUDIO_EXTS = ['.ogg', '.wav', '.mp3', '.flac']
available_files = list(TEST_DIR.glob('*.*'))
print(f'\n[DEBUG] Klasordeki Dosyalar ({len(available_files)}):')
for af in available_files[:5]:
    print(f'  - {af.name}')

def resolve_test_path(audio_id):
    base = str(audio_id)
    for ext in AUDIO_EXTS:
        p = TEST_DIR / f"{base}{ext}"
        if p.exists(): return p
    for af in available_files:
        if base in af.name or af.stem.startswith(base): return af
    audio_only = [f for f in available_files if f.suffix in AUDIO_EXTS]
    if len(audio_only) == 1: return audio_only[0]
    return None

def predict_window_optimized(audio_id, filepath, end_sec, models):
    all_probs = []
    try:
        slice_audio = get_audio_slice(audio_id, filepath, end_sec)
        mel_t = slice_to_mel_tensor(slice_audio)
        for m in models:
            with torch.no_grad():
                logits = m(mel_t)
                probs = torch.sigmoid(logits).numpy()[0]
            all_probs.append(probs)
        return np.mean(all_probs, axis=0) if all_probs else np.zeros(NUM_CLASSES)
    except Exception as e:
        print(f"Error predicting window: {e}")
        return np.zeros(NUM_CLASSES)

for idx, row in tqdm(sub.iterrows(), total=len(sub), desc='Predicting'):
    row_id = str(row['row_id'])
    parts = row_id.split('_')
    audio_id = '_'.join(parts[:-1])
    try:
        end_sec = int(parts[-1])
    except:
        end_sec = 5 
        
    filepath = resolve_test_path(audio_id)
    
    if filepath is not None and filepath.exists():
        probs = predict_window_optimized(audio_id, str(filepath), end_sec, ensemble)
        for sp in species_cols:
            sub.loc[idx, sp] = probs[species2idx[sp]]
    else:
        for sp in species_cols:
            sub.loc[idx, sp] = 0.0

sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'\nSubmission saved!')
print(sub.head())
print(f'\nShape: {sub.shape}')


Submission rows: 3
Columns: ['row_id', '1161364', '116570', '1176823', '1491113']...

[DEBUG] Klasordeki Dosyalar (1):
  - readme.txt


Predicting: 100%|██████████| 3/3 [00:00<00:00, 32.45it/s]


Submission saved!
                                    row_id  1161364  116570  1176823  \
0   BC2026_Test_0001_S05_20250227_010002_5      0.0     0.0      0.0   
1  BC2026_Test_0001_S05_20250227_010002_10      0.0     0.0      0.0   
2  BC2026_Test_0001_S05_20250227_010002_15      0.0     0.0      0.0   

    1491113  1595929  209233  22930  22956  22961  ...  whnjay1  whtdov  \
0  0.004274      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0   
1  0.004274      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0   
2  0.004274      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0   

   whwpic1  y00678  yebcar  yebela1  yecmac  yecpar  yehcar1  yeofly1  
0      0.0     0.0     0.0      0.0     0.0     0.0      0.0      0.0  
1      0.0     0.0     0.0      0.0     0.0     0.0      0.0      0.0  
2      0.0     0.0     0.0      0.0     0.0     0.0      0.0      0.0  

[3 rows x 235 columns]

Shape: (3, 235)
